In [0]:
%run ../utils/__utils__

In [0]:
config = get_config()
catalog = config["catalog"]
percorso_raw = config["percorso_raw"]
tabella = f"{catalog}.bronze.co2_emissions_canada"
print(f"Leggo: {percorso_raw}")
print(f"Scrivo in: {tabella}")

In [0]:
import re
from pyspark.sql import functions as F

def pulisci_nome(nome):
    """'Engine Size(L)' -> 'engine_size_l'"""
    return re.sub(r"[^a-z0-9]+", "_", nome.lower()).strip("_")

def carica_csv():
    df = (spark.read
          .option("header", True)        # la prima riga sono i nomi delle colonne
          .option("inferSchema", True)   # Spark capisce da solo numeri e testi
          .csv(percorso_raw))
    df = df.toDF(*[pulisci_nome(c) for c in df.columns])
    df = df.withColumn("data_caricamento", F.current_timestamp())
    (df.write
       .mode("overwrite")                 # idempotente: sostituisce, non aggiunge
       .option("overwriteSchema", "true")
       .saveAsTable(tabella))

def controlla_risultato():
    righe = spark.table(tabella).count()
    print(f"La tabella {tabella} contiene {righe} righe")
    print(spark.table(tabella).columns)

In [0]:
steps_list = [
    {"name": "carica_csv_in_bronze", "query": carica_csv},
    {"name": "controlla_risultato", "query": controlla_risultato},
]

run_query_send_logs(steps_list)